In [0]:
from pyspark.sql import functions as f, Window
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
df_member_history = spark.sql(f"""
    SELECT 
        CAST(MBRSHP_HIST_SID AS LONG) AS MBRSHP_HIST_SID
        ,CAST(MBRSHP_SID AS LONG) AS MBRSHP_SID
        ,CAST(EFF_DT AS DATE) AS EFF_DT
        ,CAST(EXP_DT AS DATE) AS EXP_DT
        ,MBRSHP_STAT_CD
        ,CAST(MBRSHP_EXP_DT AS DATE) AS MBRSHP_EXP_DT
        ,CAST(MBRSHP_RNWL_DT AS DATE) AS MBRSHP_RNWL_DT
        ,CAST(MBRSHP_FEE_INC AS double) AS MBRSHP_FEE_INC
        ,RWDS_MBR_IND
        ,CAST(RWDS_MBR_ENR_DT AS DATE) AS RWDS_MBR_ENR_DT
        ,CAST(CLUB_OF_FREQUENCY AS INT) AS CLUB_OF_FREQUENCY
        ,TEAM_MBR_IND
    FROM 
        {bronze_master_member_history}
""")

w = Window.partitionBy("MBRSHP_SID", "EFF_DT").orderBy(
    f.col("MBRSHP_HIST_SID").desc_nulls_last()
)
df_member_history = df_member_history.withColumn("rank", f.row_number().over(w))
df_member_history = df_member_history.filter(df_member_history.rank == 1).drop("rank").drop("MBRSHP_HIST_SID")

df_member_history = df_member_history.withColumn("DIFF", f.datediff(df_member_history.EXP_DT, df_member_history.EFF_DT))
df_member_history = df_member_history.filter(df_member_history.DIFF > 0)

weird = df_member_history.groupby("MBRSHP_SID", "EFF_DT").count()
df_member_history = df_member_history.join(weird, ["MBRSHP_SID", "EFF_DT"], "left_outer")
micro_cases = df_member_history.filter(
    (df_member_history["count"] > 1) & (df_member_history.CLUB_OF_FREQUENCY.isNotNull())
)
df_member_history = df_member_history.filter(df_member_history["count"] < 2)
df_member_history = df_member_history.union(micro_cases)

fiscal_days = spark.read.table(silver_fiscal_days)
df_member_history = df_member_history.join(
    fiscal_days.select("FISCAL_DAY", "FISCAL_WEEK_END"),
    df_member_history.EFF_DT == fiscal_days.FISCAL_DAY,
    "left_outer",
).drop("FISCAL_DAY", "DIFF", "count")
df_member_history = df_member_history.dropDuplicates()

df_member_history.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'member_history', config_validation, df_member_history, stats_etl_path
    )

### Merge

In [0]:
df_member_history.write.mode("overwrite").saveAsTable(silver_master_member_history)

if archive_flag:
    save_archive(df_member_history, silver_master_member_history_archive, run_as_date)